In [86]:
import psycopg2

conn = psycopg2.connect("dbname=ecommerce_sales user=postgres password=lucifer54@")
cursor = conn.cursor()

In [87]:
import pandas as pd


In [88]:
data = pd.read_csv('../data/sales_data.csv')
data.head()

,OrderDate,ProductName,Category,Region,Quantity,Sales,Profit
0,2024-12-31,Printer,Office,North,4,3640,348.93
1,2022-11-27,Mouse,Accessories,East,7,1197,106.53
2,2022-05-11,Tablet,Electronics,South,5,5865,502.73
3,2024-03-16,Mouse,Accessories,South,2,786,202.87
4,2022-09-10,Mouse,Accessories,West,1,509,103.28


Create tables


In [89]:
def create_table_regions():

  query = 'CREATE TABLE IF NOT EXISTS REGIONS(region_id int PRIMARY KEY, region_name varchar(25));'
  cursor.execute(query)
  conn.commit()
  print("Created table categories!")

create_table_regions()

Created table categories!


In [ ]:
query = 'INSERT INTO REGIONS(region_id,region_name)VALUES(%s,%s);'

cursor.executemany(query,([1,"North"],[2,"South"],[3,"West"],[4,"East"]))
conn.commit()

In [91]:
def create_table_categories():

  query = 'CREATE TABLE IF NOT EXISTS CATEGORIES(category_id int PRIMARY KEY, category_name varchar(25));'
  cursor.execute(query)
  conn.commit()
  print("Created table categories!")

create_table_categories()

Created table categories!


In [92]:
categories = set(data['Category'])
print(categories)


query = 'INSERT INTO CATEGORIES(category_id,category_name)VALUES(%s,%s);'

cursor.executemany(query,([1,"Office"],[2,"Electronics"],[3,"Accessories"]))
conn.commit()

{'Electronics', 'Office', 'Accessories'}


In [93]:
def create_table_products():

  query = 'CREATE TABLE IF NOT EXISTS PRODUCTS(product_id int PRIMARY KEY, product_name varchar(25), category_id int, FOREIGN KEY (category_id) REFERENCES CATEGORIES(category_id));'
  cursor.execute(query)
  conn.commit()
  print("Created table Products!")

create_table_products()

Created table Products!


In [94]:
product_names = list(set(data['ProductName']))


hash = {1: ['Printer'],2: ['Mouse','Keyboard'],3: ['Camera', 'Tablet', 'Headphones', 'Laptop', 'Monitor','Smartphone', 'Smartwatch']}


print(product_names)
print(categories)
products = []
for i in range(len(product_names)):
  cat = ''
  for key,val in hash.items():
    if product_names[i] in val:
      cat = key
      break
  pdt = (i+1,product_names[i],cat)
  products.append(pdt)
print(products)



['Headphones', 'Smartphone', 'Laptop', 'Printer', 'Monitor', 'Tablet', 'Keyboard', 'Camera', 'Smartwatch', 'Mouse']
{'Electronics', 'Office', 'Accessories'}
[(1, 'Headphones', 3), (2, 'Smartphone', 3), (3, 'Laptop', 3), (4, 'Printer', 1), (5, 'Monitor', 3), (6, 'Tablet', 3), (7, 'Keyboard', 2), (8, 'Camera', 3), (9, 'Smartwatch', 3), (10, 'Mouse', 2)]


In [95]:
query = "INSERT INTO PRODUCTS(product_id,product_name,category_id)VALUES(%s,%s,%s);"

cursor.executemany(query,products)
conn.commit()

In [96]:
def create_table_sales():

  query = 'CREATE TABLE IF NOT EXISTS SALES(sales_id SERIAL PRIMARY KEY, p_id int, r_id int,order_date DATE, quantity int, sales int,profit DOUBLE PRECISION, FOREIGN KEY(p_id) REFERENCES PRODUCTS(product_id), FOREIGN KEY(r_id) REFERENCES REGIONS(region_id));'
  cursor.execute(query)
  conn.commit()
  print("Created table Sales!")

create_table_sales()

Created table Sales!


Sales -> p_id, r_id, order_date, quantity, sales, profit

In [128]:
cnt = 0
rows = []
for row in data.itertuples(index=False):
  order_date = row[0]
  p_name = row[1]
  query = 'SELECT product_id from PRODUCTS where product_name = %s'
  cursor.execute(query,(p_name,))
  p_id = cursor.fetchone()[0]
  reg = row[3]
  query = 'SELECT region_id from REGIONS where region_name = %s'
  cursor.execute(query,(reg,))
  r_id = cursor.fetchone()[0]
  quant = row[4]
  sales = row[5]
  profit = row[6]
  x = (p_id,r_id,order_date,quant,sales,profit)
  rows.append(x)

  
    

In [131]:
query = "INSERT INTO SALES(p_id,r_id,order_date,quantity,sales,profit)VALUES(%s,%s,%s,%s,%s,%s);"
cursor.executemany(query,rows)
conn.commit()